# 🔍 Advanced RAG Patterns

## Learning Objectives
In this notebook, you will learn:
1. **Multi-Query Retrieval** - use an LLM to generate multiple phrasings of a question so retrieval recall improves
2. **Contextual Compression** - extract only the query-relevant sentences from retrieved chunks instead of returning full documents
3. **Hybrid Search (Ensemble Retrieval)** - combine keyword search (BM25) with semantic vector search for more robust results
4. **Parent-Document Retrieval** - search over small chunks for precision while returning larger parent chunks for context
5. **End-to-End Advanced RAG Chain** - compose multi-query retrieval, compression, and generation into a single pipeline

## Prerequisites
- Foundational RAG concepts (indexing, retrieval, generation) - see `04_Retrieval_and_RAG/Introduction_to_RAG`
- An `OPENAI_API_KEY` set in a `.env` file at the project root (used for chat completions and embeddings)
- Familiarity with LangChain retrievers, `Document` objects, and LCEL (`|`) runnables

---
## ⚙️ Setup: Imports, Logging & Sample Data

This section imports the retrievers we'll compare (multi-query, contextual compression, ensemble/BM25, and parent-document), configures logging so we can watch `MultiQueryRetriever` generate its query variations, and defines two small in-memory corpora that the demos below retrieve from.

> **Note**: The retriever classes here (`MultiQueryRetriever`, `ContextualCompressionRetriever`, `EnsembleRetriever`, `ParentDocumentRetriever`) live in `langchain_classic` - the package that now hosts LangChain's legacy retrieval abstractions on LangChain 1.x.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and Logging Configuration
# ============================================================================
from dotenv import load_dotenv
import logging

from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

# Enable logging to see multi-query generation
logging.basicConfig(level=logging.INFO, format="%(name)s - %(message)s")
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

print("✅ Imports loaded and logging configured!")
print("🔍 Multi-query retriever logs will appear as INFO-level output below its demo cell.")

C:\Users\soura\AppData\Local\Temp\ipykernel_24544\3916973658.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


✅ Imports loaded and logging configured!
🔍 Multi-query retriever logs will appear as INFO-level output below its demo cell.


### 📄 Sample Corpora

`INFO_BURIED` holds two long, narrative-style documents (a company overview and technical docs) where the relevant facts are buried among a lot of unrelated prose - useful for stress-testing retrieval, though none of the demos below query it directly yet. `TECH_DOCS` is the small, topic-tagged knowledge base that every demo function retrieves from via `create_base_vectorstore()`.

In [2]:
# ============================================================================
# SAMPLE DATA: Knowledge Base Documents
# ============================================================================
INFO_BURIED = [
    Document(
        page_content="""ACME AI SOLUTIONS - COMPANY HISTORY AND TECHNOLOGY STACK

Founded in 2018 by three Stanford graduates, ACME AI Solutions began as a
small consulting firm helping enterprises adopt machine learning. Our first
office was a converted garage in Palo Alto, and we had just two laptops and
a dream. The early days were challenging - we survived on instant ramen and
the occasional pizza from the client meetings.

In 2019, we secured our first major contract with a Fortune 500 retailer,
helping them build a recommendation engine. This led to rapid growth and we
moved to a proper office space in San Francisco. By 2020, we had grown to
50 employees and opened offices in Austin and Seattle.

Our current technology stack has evolved significantly over the years. For
backend services, we use Python and FastAPI. Our data pipeline runs on
Apache Spark and Airflow. For frontend, we've standardized on React and
TypeScript.

LangChain is a framework for building LLM applications. It provides tools
for prompts, chains, agents, and memory. LangChain supports multiple LLM
providers including OpenAI, Anthropic, and local models like Llama.

The company culture at ACME emphasizes work-life balance. We offer unlimited
PTO, which most employees use for an average of 25 days per year. Our
engineering teams follow agile methodology with two-week sprints.

Our revenue has grown consistently, from $2M in 2019 to $45M in 2023. We
project $70M for 2024, driven by our new enterprise AI platform. The company
went through Series B funding in 2022, raising $80M at a $500M valuation.

Employee benefits include comprehensive health insurance through Aetna, a
401(k) with 4% matching, and a generous equity package.""",
        metadata={"source": "acme_company_overview.pdf"},
    ),
    Document(
        page_content="""ACME AI PLATFORM - TECHNICAL DOCUMENTATION v2.4

Chapter 1: System Architecture Overview

The ACME AI Platform is built on a microservices architecture deployed on
AWS EKS (Elastic Kubernetes Service). Each microservice is containerized
using Docker and orchestrated by Kubernetes. We use Istio as our service
mesh for traffic management and observability.

Our database layer consists of PostgreSQL for transactional data, Redis
for caching, and Pinecone for vector storage. All databases are deployed
in high-availability configurations with automatic failover.

Chapter 2: Authentication and Authorization

User authentication is handled through Auth0, supporting both SSO via SAML
2.0 and OAuth 2.0 flows. We implement role-based access control (RBAC) with
four default roles: Admin, Developer, Analyst, and Viewer.

Chapter 3: AI Framework Integration

LangGraph is a library for building stateful, multi-actor applications with
LLMs. Key features include state management, cycles and loops, human-in-the-
loop workflows, and persistence. LangGraph extends LangChain for complex
agent architectures.

Chapter 4: Monitoring and Logging

We use DataDog for application performance monitoring (APM) and log
aggregation. All services emit structured JSON logs that are collected and
indexed for searching. Alert thresholds are configured for latency (p99 >
500ms), error rates (> 1%), and resource utilization (CPU > 80%).

Chapter 5: Disaster Recovery

Our disaster recovery plan includes daily database backups stored in S3
with cross-region replication. RTO is 4 hours, and RPO is 1 hour.""",
        metadata={"source": "technical_docs_v2.4.pdf"},
    ),
]

# Sample knowledge base for demos
TECH_DOCS = [
    Document(
        page_content="Python is a high-level programming language known for its simplicity and readability. It supports multiple programming paradigms including procedural, object-oriented, and functional programming. Python is widely used in web development, data science, artificial intelligence, and automation.",
        metadata={
            "topic": "programming",
            "language": "python",
            "difficulty": "beginner",
        },
    ),
    Document(
        page_content="JavaScript is the language of the web. It runs in browsers and on servers with Node.js. Modern frameworks like React, Vue, and Angular make building interactive web applications efficient. JavaScript supports asynchronous programming with Promises and async/await.",
        metadata={
            "topic": "programming",
            "language": "javascript",
            "difficulty": "intermediate",
        },
    ),
    Document(
        page_content="Machine learning is a subset of AI that enables systems to learn from data. Supervised learning uses labeled data, while unsupervised learning finds patterns in unlabeled data. Popular ML frameworks include TensorFlow, PyTorch, and scikit-learn.",
        metadata={
            "topic": "ai",
            "subtopic": "machine_learning",
            "difficulty": "advanced",
        },
    ),
    Document(
        page_content="LangChain is a framework for building LLM applications. It provides tools for prompts, chains, agents, and memory. LangChain supports multiple LLM providers including OpenAI, Anthropic, and local models.",
        metadata={
            "topic": "ai",
            "subtopic": "llm_frameworks",
            "difficulty": "intermediate",
        },
    ),
    Document(
        page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs. Key features include state management, cycles and loops, human-in-the-loop workflows, and persistence. LangGraph extends LangChain for complex agent architectures.",
        metadata={
            "topic": "ai",
            "subtopic": "llm_frameworks",
            "difficulty": "advanced",
        },
    ),
    Document(
        page_content="Docker is a platform for containerizing applications. Containers package code and dependencies together for consistent deployment. Docker Compose orchestrates multi-container applications. Kubernetes scales Docker containers in production.",
        metadata={
            "topic": "devops",
            "subtopic": "containers",
            "difficulty": "intermediate",
        },
    ),
    Document(
        page_content="PostgreSQL is an advanced open-source relational database. It supports JSON data types, full-text search, and extensions like pgvector for vector similarity search. PostgreSQL is ACID compliant and highly extensible.",
        metadata={
            "topic": "database",
            "type": "relational",
            "difficulty": "intermediate",
        },
    ),
    Document(
        page_content="Vector databases like Pinecone, Chroma, and Qdrant are optimized for storing and searching embeddings. They enable semantic similarity search for RAG applications. Most support metadata filtering and hybrid search combining keywords with vectors.",
        metadata={"topic": "database", "type": "vector", "difficulty": "intermediate"},
    ),
]

print(f"✅ Sample data ready: INFO_BURIED ({len(INFO_BURIED)} docs), TECH_DOCS ({len(TECH_DOCS)} docs)")

✅ Sample data ready: INFO_BURIED (2 docs), TECH_DOCS (8 docs)


---
## 🧱 Shared Helper: Base Vector Store

Every demo below needs a vector store over `TECH_DOCS`. This helper builds a fresh Chroma collection from those documents using OpenAI embeddings, so each demo function can call it independently.

In [3]:
# ============================================================================
# CREATE_BASE_VECTORSTORE: Shared Chroma Vector Store for the Demos
# ============================================================================
# ============ CREATE_BASE_VECTORSTORE =====================================
def create_base_vectorstore():
    """Create a basic vector store for demos."""
    return Chroma.from_documents(
        documents=TECH_DOCS,
        embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    )

---
## 🔀 Part 1: Multi-Query Retriever

A single query often misses relevant documents just because it's phrased differently than the source text. `MultiQueryRetriever` asks an LLM to generate several alternative phrasings of the user's question, retrieves for each, and de-duplicates the union of results.

### Key Insight:
> More query perspectives -> higher recall, at the cost of extra LLM calls and retrieval round-trips per question.

In [4]:
# ============================================================================
# DEMO: Multi-Query Retriever
# ============================================================================
# ============ DEMO_MULTI_QUERY_RETRIEVER ==================================
def demo_multi_query_retriever():
    """Multi-Query Retriever generates multiple query perspectives."""

    print("=" * 60)
    print("MULTI-QUERY RETRIEVER")
    print("Generates multiple perspectives on your question")
    print("=" * 60)

    vectorstore = create_base_vectorstore()
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

    # Create multi-query retriever
    retriever = MultiQueryRetriever.from_llm(
        retriever=vectorstore.as_retriever(search_kwargs={"k": 2}), llm=llm
    )

    query = "What tools can I use to build AI applications?"

    print(f"\nOriginal Query: {query}")
    print("\nThe retriever will generate multiple query variations...")
    print("(Check INFO logs above for generated queries)\n")

    # Retrieve documents
    docs = retriever.invoke(query)

    print(f"Retrieved {len(docs)} unique documents:")
    for i, doc in enumerate(docs):
        print(
            f"\n{i+1}. [{doc.metadata.get('topic', 'N/A')}] {doc.page_content[:100]}..."
        )

---
## ✂️ Part 2: Contextual Compression Retriever

Raw retrieved chunks often contain a mix of relevant and irrelevant sentences. `ContextualCompressionRetriever` wraps a base retriever with an `LLMChainExtractor` that reads each retrieved chunk against the query and keeps only the parts that actually answer it - shrinking what gets passed to the generation step.

In [5]:
# ============================================================================
# DEMO: Contextual Compression Retriever
# ============================================================================
# ============ DEMO_CONTEXTUAL_COMPRESSION =================================
def demo_contextual_compression():
    """Contextual Compression extracts only relevant parts."""

    print("=" * 60)
    print("CONTEXTUAL COMPRESSION RETRIEVER")
    print("Extracts only query-relevant content from documents")
    print("=" * 60)

    vectorstore = create_base_vectorstore()
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # Create compressor
    compressor = LLMChainExtractor.from_llm(llm)

    # Wrap retriever with compression
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    )

    query = "What frameworks exist for building LLM applications?"

    print(f"\nQuery: {query}")

    # Without compression
    base_docs = vectorstore.as_retriever(search_kwargs={"k": 2}).invoke(query)
    print(f"\n--- WITHOUT Compression (full chunks) ---")
    for doc in base_docs:
        print(f"Length: {len(doc.page_content)} chars")
        print(f"Content: {doc.page_content[:150]}...\n")

    # With compression
    # With compression
    compressed_docs = compression_retriever.invoke(query)
    print(f"\n--- WITH Compression (relevant only) ---")
    for doc in compressed_docs:
        print(f"Length: {len(doc.page_content)} chars")
        print(f"Content: {doc.page_content}\n")

---
## 🔎 Part 3: Ensemble / Hybrid Search (BM25 + Semantic)

Keyword search (BM25) excels at exact terms and acronyms; semantic search excels at conceptual/paraphrased queries. `EnsembleRetriever` runs both retrievers and merges their ranked results using weighted Reciprocal Rank Fusion, weighted here 40% BM25 / 60% semantic.

In [6]:
# ============================================================================
# DEMO: Ensemble (Hybrid) Retriever
# ============================================================================
# ============ DEMO_ENSEMBLE_HYBRID_SEARCH =================================
def demo_ensemble_hybrid_search():
    """Hybrid search combining keyword (BM25) and semantic search."""

    print("=" * 60)
    print("ENSEMBLE/HYBRID RETRIEVER")
    print("Combines keyword (BM25) + semantic search")
    print("=" * 60)

    vectorstore = create_base_vectorstore()

    # BM25 keyword retriever
    bm25_retriever = BM25Retriever.from_documents(TECH_DOCS)
    bm25_retriever.k = 3

    # Semantic retriever
    semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # Ensemble combines both
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, semantic_retriever],
        weights=[0.4, 0.6],  # 40% keyword, 60% semantic
    )

    # Test queries
    # queries = [
    #     "PostgreSQL pgvector",  # Keyword-heavy (BM25 helps)
    #     "What database stores embeddings?",  # Semantic (vectors help)
    # ]
    queries = [
        "ACID transactions",  # Keyword-heavy (BM25 helps)
        "How do I store AI model outputs for later retrieval?",  # Semantic (vectors help)
        "fast similarity lookup for embeddings",  # Mixed
    ]

    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 40)

        # Compare results
        bm25_results = bm25_retriever.invoke(query)
        semantic_results = semantic_retriever.invoke(query)
        ensemble_results = ensemble_retriever.invoke(query)

        print(f"BM25 top result: {bm25_results[0].page_content[:60]}...")
        print(f"Semantic top result: {semantic_results[0].page_content[:60]}...")
        print(f"Ensemble top result: {ensemble_results[0].page_content[:60]}...")

---
## 🌳 Part 4: Parent-Document Retriever

Small chunks are best for precise similarity search, but they lack surrounding context when handed to the LLM. `ParentDocumentRetriever` searches over small child chunks, then returns the larger parent chunk each child came from - combining search precision with generation-time context.

In [7]:
# ============================================================================
# DEMO: Parent-Document Retriever
# ============================================================================
# ============ DEMO_PARENT_DOCUMENT_RETRIEVER ==============================
def demo_parent_document_retriever():
    """Parent Document Retriever: small chunks for search, large for context."""

    print("=" * 60)
    print("PARENT DOCUMENT RETRIEVER")
    print("Small chunks for precise search, large chunks for context")
    print("=" * 60)
    # Long document to demonstrate parent/child splitting
    long_doc = Document(
        page_content="""
# Complete Guide to Building AI Agents

## Chapter 1: Introduction to AI Agents

AI agents are autonomous systems that can perceive their environment, make decisions, and take actions to achieve goals. Unlike simple chatbots, agents can use tools, maintain state, and execute multi-step plans.

The key components of an AI agent include:
- A language model for reasoning
- Tools for interacting with external systems
- Memory for maintaining context
- A planning mechanism for complex tasks

## Chapter 2: Agent Frameworks

Several frameworks exist for building AI agents:

LangChain provides the foundational abstractions for chains and simple agents. It excels at straightforward tool-calling patterns and integrates with many LLM providers.

LangGraph extends LangChain for complex, stateful agents. It introduces graph-based state management, enabling cycles, human-in-the-loop workflows, and persistent execution.

CrewAI focuses on multi-agent collaboration, allowing teams of specialized agents to work together on complex tasks.

## Chapter 3: Production Considerations

Deploying agents to production requires careful attention to:
- Error handling and fallbacks
- Token usage optimization
- Observability and tracing
- Security and access control
- State persistence and recovery

LangSmith provides observability for LangChain/LangGraph applications, offering tracing, evaluation, and monitoring capabilities.
        """,
        metadata={"source": "ai_agents_guide.md"},
    )

    # Splitters
    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

    # Storage
    vectorstore = Chroma(
        collection_name="parent_child_demo",
        embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    )
    store = InMemoryStore()

    # Create retriever
    retriever = ParentDocumentRetriever(
        vectorstore=vectorstore,
        docstore=store,
        child_splitter=child_splitter,
        parent_splitter=parent_splitter,
    )

    # Add document
    retriever.add_documents([long_doc])

    # Search
    query = "What is LangGraph used for?"

    print(f"\nQuery: {query}")

    # Regular retrieval (would get small chunks)
    child_docs = vectorstore.similarity_search(query, k=1)
    print(f"\n--- Child Chunk (what search found) ---")
    print(f"Length: {len(child_docs[0].page_content)} chars")
    print(f"Content: {child_docs[0].page_content}")

    # Parent retrieval (gets full context)
    parent_docs = retriever.invoke(query)
    print(f"\n--- Parent Chunk (what's returned) ---")
    print(f"Length: {len(parent_docs[0].page_content)} chars")
    print(f"Content preview: {parent_docs[0].page_content[:300]}...")

---
## 🔗 Part 5: Complete Advanced RAG Chain

This demo composes the techniques above into one pipeline: `MultiQueryRetriever` for recall, wrapped in `ContextualCompressionRetriever` for precision, feeding a prompt template and the LLM through an LCEL chain (`{{context, question}} | prompt | llm | StrOutputParser()`).

In [8]:
# ============================================================================
# DEMO: Complete Advanced RAG Chain (Multi-Query + Compression + Generation)
# ============================================================================
# ============ DEMO_ADVANCED_RAG_CHAIN =====================================
def demo_advanced_rag_chain():
    """Complete RAG chain with advanced retrieval."""

    print("=" * 60)
    print("COMPLETE ADVANCED RAG CHAIN")
    print("Multi-query + Compression + RAG")
    print("=" * 60)

    vectorstore = create_base_vectorstore()
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # Multi-query for better recall
    multi_retriever = MultiQueryRetriever.from_llm(
        retriever=vectorstore.as_retriever(search_kwargs={"k": 3}), llm=llm
    )

    # Compression to focus on relevant info
    compressor = LLMChainExtractor.from_llm(llm)
    advanced_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=multi_retriever
    )

    # RAG prompt
    prompt = ChatPromptTemplate.from_template(
        """
Answer the question based on the following context. Be specific and cite which technologies you're referring to.

Context:
{context}

Question: {question}

Answer:"""
    )

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # Build chain
    rag_chain = (
        {"context": advanced_retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # Test
    questions = [
        "What options do I have for building AI agents?",
        "How can I store and search embeddings?",
    ]

    for q in questions:
        print(f"\nQ: {q}")
        answer = rag_chain.invoke(q)
        print(f"A: {answer}")

---
## ▶️ Running the Demos

The original script's `if __name__ == "__main__":` guard is kept verbatim below - Jupyter sets `__name__` to `"__main__"`, so this cell runs as-is. Uncomment any of the other demo calls to run that pattern instead (each performs live LLM/embedding calls, so only one is left active by default).

In [10]:
# ============================================================================
# RUN: Execute a Demo
# ============================================================================
# ============ RUN =========================================================
if __name__ == "__main__":
    # demo_multi_query_retriever()
    # demo_contextual_compression()
    # demo_ensemble_hybrid_search()
    # demo_parent_document_retriever()
    demo_advanced_rag_chain()

COMPLETE ADVANCED RAG CHAIN
Multi-query + Compression + RAG


httpx2 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



Q: What options do I have for building AI agents?


httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
langchain_classic.retrievers.multi_query - Generated queries: ['What are the different approaches I can take to create AI agents?  ', 'What methods are available for developing AI agents?  ', 'Can you provide a list of options for constructing AI agents?']
httpx2 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
httpx2 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
httpx2 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: When building AI agents using LangChain, you have several options that leverage its various components:

1. **Chains**: You can create sequences of calls to language models or other tools. Chains allow you to define a series of steps that the agent will follow to process input and generate output. For example, you might create a chain that first processes user input, then queries a database, and finally formats the response.

2. **Agents**: LangChain provides a framework for building agents that can make decisions based on user input and context. Agents can utilize tools to perform specific tasks, such as retrieving information or executing commands. You can define the logic that determines which tool to use based on the input received.

3. **Prompts**: You can design custom prompts to guide the behavior of the language model. This allows you to tailor the agent's responses and ensure that it adheres to specific guidelines or formats. By crafting effective prompts, you can enhance t

httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
langchain_classic.retrievers.multi_query - Generated queries: ['What are the best methods for storing and retrieving embeddings efficiently?  ', 'What techniques can I use to manage and query embedding data?  ', 'How do I implement a system for storing and searching through embeddings?']
httpx2 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
httpx2 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
httpx2 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
httpx2 - HTTP Req

A: To store and search embeddings, you can use specialized vector databases or extend traditional relational databases. Here are some options:

1. **Vector Databases**: 
   - **Pinecone**: This is a fully managed vector database that is optimized for storing and searching embeddings. It supports semantic similarity search, making it suitable for applications like Retrieval-Augmented Generation (RAG). Pinecone also allows for metadata filtering and hybrid search, which combines keyword searches with vector searches.
   - **Chroma**: Similar to Pinecone, Chroma is designed for managing embeddings and supports semantic similarity search. It also offers features for metadata filtering and hybrid search.
   - **Qdrant**: This vector database is optimized for real-time vector similarity search and can handle large-scale embedding storage. It supports metadata filtering and hybrid search capabilities as well.

2. **PostgreSQL with Extensions**:
   - **PostgreSQL**: This traditional relational

---
## 📝 Summary

In this notebook, we explored five advanced retrieval patterns that go beyond naive top-k similarity search:

### 1. Recall-Boosting Retrieval
- **Multi-Query Retriever**: generates multiple phrasings of a question via an LLM and unions the results
- **Ensemble/Hybrid Retriever**: blends BM25 keyword search with semantic vector search using weighted rank fusion

### 2. Precision- and Context-Boosting Retrieval
- **Contextual Compression Retriever**: uses an LLM to extract only the query-relevant sentences from each chunk
- **Parent-Document Retriever**: searches small child chunks but returns their larger parent chunks for full context

### 3. Composing an Advanced RAG Chain
- Retrieval techniques compose via LCEL: multi-query -> compression -> prompt -> LLM -> output parser
- Each technique trades off recall, precision, latency, and LLM-call cost differently - pick per use case

### Functions Defined in This Notebook
- `create_base_vectorstore()` - shared Chroma vector store over `TECH_DOCS`
- `demo_multi_query_retriever()`
- `demo_contextual_compression()`
- `demo_ensemble_hybrid_search()`
- `demo_parent_document_retriever()`
- `demo_advanced_rag_chain()`

### Next Steps
- Explore `Query_Transformation_Techniques/` for RAG-Fusion, query decomposition, HyDE, and reranking
- See `08_Advanced_RAG/` for agentic/self-correcting RAG that adds evaluation and retry loops on top of these retrievers